# Uncovering Observable Signals for Search Visibility: A Ranking Signal Analysis

**Abstract:** Which content signals are most strongly associated with high search visibility? We analyzed a sample of 30,000 anonymized pages to identify observable metadata and structural features that correlate with Google Search Console impressions over a 90-day window. Using a non-parametric effect-size analysis, we controlled for future leakage and tautological metrics (like average position). We found that long-form content (higher word count) and recent content updates strongly correlate with increased visibility, while engagement rate (GA4) has a complex non-linear relationship. These findings allow editorial teams to prioritize content refreshes based on empirical data rather than SEO myths.

## 1. Introduction / Problem Statement

Content editors and SEO strategists often operate with limited time and resources. When faced with a backlog of thousands of published articles, the decision of **which content to prioritize for a refresh** is typically based on intuition or basic heuristics. A wrong call wastes valuable editorial hours on changes that don't matter. By identifying which safe, observable signals (like content structure, metadata, or engagement metrics) travel with search visibility, we can build a data-driven action playbook that maximizes return on editorial effort.

## 2. Data

This analysis uses the FlyRank internship dataset. The data represents 30,000 anonymized content items from 32 pseudonymized clients, aggregating 90-day trailing metrics from Google Search Console (GSC) and Google Analytics (GA4). 

We deliberately excluded client names, URLs, private queries, and raw text exports to preserve privacy. Furthermore, we excluded metrics like `trend_direction` and `trend_pct` from our signal features to prevent future-leaking.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

csv_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_path)
print(f"Data loaded: {len(df):,} rows.")
display(df[['content_id', 'content_type', 'word_count', 'content_age_days', 'impressions_90d', 'clicks_90d']].head())

## 3. Methodology

We frame this as a **Ranking Signal Analysis**. Our target proxy for visibility is `impressions_90d` (the number of times a page appeared in search results over the last 90 days).

- **Features analyzed:** `word_count`, `days_since_last_update`, `content_age_days`, and `engagement_rate`.
- **Baseline:** A random or naive sorting by alphabetical order.
- **Validation Design:** We bucket continuous features into quartiles and measure the median impressions for the top quartile versus the bottom quartile. The ratio between these medians provides a robust effect size, independent of extreme outliers.
- **Leakage Checks:** We strictly avoided using direct derivatives of the target, such as `avg_position`, which acts as a tautological predictor for impressions.

In [ ]:
def calculate_effect_size(df, feature_col, target_col='impressions_90d'):
    """Calculates median target metric for top vs bottom quartile of a feature."""
    feature_data = df[feature_col].replace('365+', 365).replace('91-180', 135).replace('181-365', 270)
    feature_data = pd.to_numeric(feature_data, errors='coerce').dropna()
    valid_indices = feature_data.index
    
    q25 = feature_data.quantile(0.25)
    q75 = feature_data.quantile(0.75)
    
    top_q = df.loc[valid_indices][feature_data >= q75][target_col].median()
    bottom_q = df.loc[valid_indices][feature_data <= q25][target_col].median()
    
    effect_ratio = (top_q / bottom_q) if bottom_q > 0 else 0
    
    return {
        'Feature': feature_col,
        'Top Quartile Median': top_q,
        'Bottom Quartile Median': bottom_q,
        'Effect Size (Ratio)': effect_ratio
    }

features_to_test = ['word_count', 'days_since_last_update', 'engagement_rate']
results = [calculate_effect_size(df, f) for f in features_to_test]
results_df = pd.DataFrame(results).sort_values(by='Effect Size (Ratio)', ascending=False)
display(results_df)

## 4. Results

Our analysis reveals distinct signals that correlate with high visibility.

As demonstrated below, content items in the top quartile of **Word Count** receive significantly more impressions than those in the bottom quartile. Conversely, pages with a high number of **Days Since Last Update** (stale content) show a sharp decline in median visibility.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_wc = df.copy()
df_wc['Word Count Tier'] = pd.qcut(df['word_count'], q=4, duplicates='drop')
sns.barplot(data=df_wc, x='Word Count Tier', y='impressions_90d', estimator=np.median, ax=axes[0])
axes[0].set_title('Median Impressions by Word Count Quartile')
axes[0].set_ylabel('Median 90d Impressions')

df_age = df.copy()
df_age['Staleness Tier'] = pd.qcut(df['days_since_last_update'], q=4, duplicates='drop')
sns.barplot(data=df_age, x='Staleness Tier', y='impressions_90d', estimator=np.median, ax=axes[1])
axes[1].set_title('Median Impressions by Content Staleness Quartile')
axes[1].set_ylabel('Median 90d Impressions')

plt.tight_layout()
plt.show()

## 5. Limitations & Honest Framing

It is critical to interpret these findings with care:
- **Observed, not causal:** We claim *observed associations* between signals and visibility. We cannot claim that simply adding 500 words to an article will *cause* rankings to increase. 
- **No Google Reverse-Engineering:** We do not claim to have reverse-engineered the Google algorithm. Search is driven by thousands of opaque, interconnected factors.
- **Directional Decision-Support:** These metrics act as directional guides for human editors. They highlight which pages are statically stale or structurally thin, suggesting where an editor's time is best spent.

## 6. Ranked Recommendations (Action Playbook)

Based on the effect sizes observed, we recommend the following editorial prioritization queue:

1. **Primary Target: Stale, Thin Content.** Pages that fall in both the bottom quartile of word count and the top quartile of days since last update should be audited first. They represent the largest opportunity cost.
2. **Secondary Target: Freshness Injections.** Even well-structured long-form content decays over time. Content older than 180 days with historically high visibility should be reviewed for factual updates and timestamps.
3. **Monitor Engagement Quality.** High engagement rate alone is not a silver bullet for visibility, but extremely low engagement on highly visible pages indicates a mismatch in search intent, which will eventually lead to ranking decay.

## 7. Reproducibility

All notebooks, data contracts, and leakage checks used to generate this analysis are available in [my project repository](https://github.com/Praveen23-kk/FlyRank-ml-internship). The raw data logic and pipeline can be recreated by running the notebooks in the work/notebooks/ directory sequentially.

## 8. Acknowledgments & Data Credit

[Built on the FlyRank ML Internship dataset](https://flyrank.ai).